## Sezione 0 – Setup e Import

Installazione delle dipendenze mancanti tramite pip e importazione di tutte le librerie.
Seed globale `RANDOM_STATE = 42`. Le librerie opzionali (xgboost, lightgbm, shap, umap,
imbalanced-learn) vengono sostituite da fallback interni se non installabili.

In [ ]:
# Installazione dipendenze
import subprocess, sys

def _pip(pkg, timeout=120):
    """Installa un pacchetto; ritorna True se ha successo."""
    for extra in [[], ['--break-system-packages']]:
        try:
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', '-q', pkg] + extra,
                timeout=timeout, stderr=subprocess.DEVNULL
            )
            return True
        except Exception:
            pass
    return False

DIPENDENZE = ['pyarrow', 'xgboost', 'lightgbm', 'shap', 'imbalanced-learn']
for pkg in DIPENDENZE:
    ok = _pip(pkg)
    stato = '✅ installato' if ok else '⚠️  non installabile (fallback attivo)'
    print(f'  {pkg}: {stato}')
print('\n✅ Fase installazione completata')

In [ ]:
# ── Import librerie standard ────────────────────────────────────────────────
import os, math, re, warnings, json
from pathlib import Path
import glob as _glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, IsolationForest, VotingClassifier,
    GradientBoostingClassifier, ExtraTreesClassifier
)
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve,
    average_precision_score, roc_auc_score,
    accuracy_score, f1_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as scipy_entropy
import joblib

# ── XGBoost ──────────────────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGB_OK = True; print('✅ XGBoost disponibile')
except ImportError:
    XGB_OK = False; print('⚠️  XGBoost non disponibile — fallback GradientBoosting')
    class XGBClassifier:
        def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3,
                     subsample=1.0, random_state=42, scale_pos_weight=1,
                     use_label_encoder=False, eval_metric='logloss', **kw):
            self._m = GradientBoostingClassifier(
                n_estimators=n_estimators, learning_rate=learning_rate,
                max_depth=max_depth, subsample=subsample, random_state=random_state)
        def fit(self, X, y, **kw):  self._m.fit(X, y); return self
        def predict(self, X):       return self._m.predict(X)
        def predict_proba(self, X): return self._m.predict_proba(X)
        @property
        def feature_importances_(self): return self._m.feature_importances_

# ── LightGBM ─────────────────────────────────────────────────────────────────
try:
    from lightgbm import LGBMClassifier
    LGBM_OK = True; print('✅ LightGBM disponibile')
except ImportError:
    LGBM_OK = False; print('⚠️  LightGBM non disponibile — fallback ExtraTrees')
    class LGBMClassifier:
        def __init__(self, n_estimators=100, random_state=42,
                     learning_rate=0.1, max_depth=-1,
                     class_weight=None, verbose=-1, **kw):
            self._m = ExtraTreesClassifier(
                n_estimators=n_estimators, random_state=random_state,
                class_weight=class_weight)
        def fit(self, X, y, **kw):  self._m.fit(X, y); return self
        def predict(self, X):       return self._m.predict(X)
        def predict_proba(self, X): return self._m.predict_proba(X)
        @property
        def feature_importances_(self): return self._m.feature_importances_

# ── SHAP ─────────────────────────────────────────────────────────────────────
try:
    import shap; SHAP_OK = True; print('✅ SHAP disponibile')
except ImportError:
    SHAP_OK = False; print('⚠️  SHAP non disponibile — sezione SHAP saltata')

# ── UMAP ─────────────────────────────────────────────────────────────────────
try:
    import umap; UMAP_OK = True; print('✅ UMAP disponibile')
except ImportError:
    UMAP_OK = False; print('⚠️  UMAP non disponibile — si usa PCA 2D')

# ── SMOTE ────────────────────────────────────────────────────────────────────
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_OK = True; print('✅ SMOTE disponibile')
except ImportError:
    SMOTE_OK = False; print('⚠️  SMOTE non disponibile — solo class_weight per sbilanciamento')

# ── Configurazione globale ────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 120

OUTPUT_DIR = Path('/home/cesare/Scrivania/Vannacci/mib_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'\n📁 Directory output: {OUTPUT_DIR}')
print(f'🌱 RANDOM_STATE = {RANDOM_STATE}')
print('✅ Setup completato')

## Sezione 1 – Ingest e Ispezione

Caricamento di `Mib_final.parquet`. Stampa di shape, tipi di dato, prime righe,
conteggio valori nulli e distribuzione di `is_bot`. Identificazione automatica
delle colonne chiave e della granularità (tweet-level vs user-level).

In [ ]:
print('⏳ Inizio: caricamento Mib_final.parquet...')

# ── Rilevamento percorso ──────────────────────────────────────────────────────
PERCORSO_PRINCIPALE = Path('/home/cesare/Scrivania/Vannacci/Mib_final.parquet')

if PERCORSO_PRINCIPALE.exists():
    PERCORSO_PARQUET = PERCORSO_PRINCIPALE
else:
    print(f'⚠️  File non trovato in {PERCORSO_PRINCIPALE}, ricerca automatica...')
    candidati = (
        _glob.glob('/home/**/*.parquet', recursive=True) +
        _glob.glob('./**/*.parquet',     recursive=True)
    )
    mib = [c for c in candidati if 'mib' in c.lower()]
    if mib:
        PERCORSO_PARQUET = Path(mib[0])
        print(f'✅ Trovato: {PERCORSO_PARQUET}')
    else:
        raise FileNotFoundError(
            'Impossibile trovare Mib_final.parquet. '
            'Assicurarsi che il file esista nel percorso atteso.'
        )

df_raw = pd.read_parquet(PERCORSO_PARQUET)
print(f'✅ Completato: {df_raw.shape[0]:,} righe × {df_raw.shape[1]} colonne')

In [ ]:
# ── Ispezione schema ─────────────────────────────────────────────────────────
print('=' * 60)
print(f'SHAPE: {df_raw.shape}')
print('=' * 60)

print('\n📌 TIPI DI DATO:')
print(df_raw.dtypes.to_string())

print('\n📌 PRIME 3 RIGHE:')
display(df_raw.head(3))

print('\n📌 VALORI NULLI PER COLONNA:')
null_c = df_raw.isnull().sum()
print(null_c[null_c > 0].to_string() if null_c.sum() > 0 else 'Nessun valore nullo')

# ── Ricerca colonna target ────────────────────────────────────────────────────
def _trova(df, varianti):
    """Cerca una colonna tra le varianti (case-insensitive)."""
    for v in varianti:
        for c in df.columns:
            if c.lower() == v.lower():
                return c
    return None

COL_BOT      = _trova(df_raw, ['is_bot','isbot','label','bot','class','target','is_automated'])
COL_USER_ID  = _trova(df_raw, ['user_id','userid','uid','author_id','user'])
COL_TEXT     = _trova(df_raw, ['text','tweet','content','message','full_text'])
COL_TIME     = _trova(df_raw, ['created_at','timestamp','date','datetime','time'])
COL_URLS     = _trova(df_raw, ['urls','url','entities_urls'])
COL_RETWEET  = _trova(df_raw, ['retweeted','is_retweet','retweet_count','rt'])
COL_FOLLOWER = _trova(df_raw, ['followers_count','followers','follower_count'])
COL_FOLLOWING= _trova(df_raw, ['friends_count','following','following_count','friends'])

if COL_BOT is None:
    raise ValueError(f'Colonna target non trovata. Colonne: {list(df_raw.columns)}')

print(f'\n✅ Colonna target: \'{COL_BOT}\'')
print('\n📌 DISTRIBUZIONE TARGET:')
dist = df_raw[COL_BOT].value_counts()
print(dist)
rapporto_bot_raw = dist.get(1, 0) / len(df_raw)
print(f'\n   Bot:   {rapporto_bot_raw:.2%}')
print(f'   Umani: {1-rapporto_bot_raw:.2%}')

print('\n📌 COLONNE CHIAVE:')
for nome, col in {
    'user_id': COL_USER_ID, 'text': COL_TEXT, 'created_at': COL_TIME,
    'urls': COL_URLS, 'retweet': COL_RETWEET,
    'followers': COL_FOLLOWER, 'following': COL_FOLLOWING
}.items():
    print(f"   {nome:15s} → {f'{chr(39)}{col}{chr(39)}' if col else 'NON TROVATA'}")

# ── Granularità ───────────────────────────────────────────────────────────────
if COL_USER_ID is not None:
    n_utenti = df_raw[COL_USER_ID].nunique()
    n_righe  = len(df_raw)
    rpu = n_righe / n_utenti
    IS_ROW_LEVEL = rpu > 2.0
    print(f'\n📌 GRANULARITÀ:')
    print(f'   {n_utenti:,} utenti, {n_righe:,} righe → {rpu:.1f} righe/utente')
    print(f'   Tipo: {"ROW-LEVEL (tweet per riga)" if IS_ROW_LEVEL else "USER-LEVEL (un record per utente)"}')
else:
    IS_ROW_LEVEL = False
    print('\n⚠️  Nessuna colonna user_id trovata — si assume user-level')

# Rinomina colonna target per uniformità
df_raw = df_raw.rename(columns={COL_BOT: 'is_bot'})
COL_BOT = 'is_bot'
print('\n✅ Ispezione completata')

## Sezione 2 – Aggregazione per Utente (User Profiling)

Se i dati sono row-level (un tweet per riga), aggrega tutti i tweet di ciascun
utente in un unico record. Vengono calcolate feature **testuali**, **comportamentali**,
**social** e di **coerenza sequenziale** (similarità coseno tra tweet consecutivi).
Se i dati sono già user-level, usa le colonne numeriche disponibili.

In [ ]:
# ── Funzioni di feature engineering ─────────────────────────────────────────

def _entropia_shannon(testo: str) -> float:
    """Entropia di Shannon su bigrammi di caratteri (misura rigidità sintattica)."""
    if not testo or len(testo) < 2:
        return 0.0
    bg = [testo[i:i+2] for i in range(len(testo)-1)]
    _, cnt = np.unique(bg, return_counts=True)
    p = cnt / cnt.sum()
    return float(-np.sum(p * np.log2(p + 1e-12)))

def _ttr(testo: str) -> float:
    """Type-Token Ratio: vocabolario unico / parole totali."""
    parole = testo.lower().split()
    return len(set(parole)) / max(len(parole), 1)

def _ratio_prefisso(testi: list, prefisso: str) -> float:
    """Frazione media di token che iniziano con `prefisso`."""
    vals = []
    for t in testi:
        tok = t.split()
        if tok:
            vals.append(sum(1 for x in tok if x.startswith(prefisso)) / len(tok))
    return float(np.mean(vals)) if vals else 0.0

def _non_ascii(testo: str) -> float:
    if not testo: return 0.0
    return sum(1 for c in testo if ord(c) > 127) / len(testo)

def _entropia_oraria(ore: list) -> float:
    """Entropia normalizzata della distribuzione nelle 24 ore."""
    if not ore: return 0.0
    cnt = np.zeros(24)
    for h in ore:
        cnt[int(h) % 24] += 1
    p = cnt / cnt.sum()
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p))) / np.log2(24)

def _cv_iat(ts_sorted: list) -> float:
    """Coefficiente di Variazione degli inter-arrival times."""
    if len(ts_sorted) < 2: return 0.0
    delta = np.diff([t.timestamp() for t in ts_sorted])
    delta = delta[delta >= 0]
    mu = np.mean(delta)
    return float(np.std(delta) / mu) if mu > 0 else 0.0

def _coerenza_coseno(testi: list, vec) -> tuple:
    """Media e std della cosine similarity tra tweet consecutivi (via TF-IDF)."""
    if len(testi) < 2: return (1.0, 0.0)
    puliti = [str(t) if t else '' for t in testi]
    try:
        V = vec.transform(puliti)
        sims = [cosine_similarity(V[i], V[i+1])[0, 0] for i in range(len(puliti)-1)]
        return (float(np.mean(sims)), float(np.std(sims)))
    except Exception:
        return (0.0, 0.0)

print('✅ Funzioni di feature engineering definite')

In [ ]:
print('⏳ Inizio: aggregazione per utente...')

if IS_ROW_LEVEL and COL_USER_ID is not None:

    # ── Preparazione ─────────────────────────────────────────────────────────
    if COL_TEXT is not None:
        df_raw[COL_TEXT] = df_raw[COL_TEXT].fillna('').astype(str)
    else:
        COL_TEXT = 'text'
        df_raw[COL_TEXT] = ''
        print('⚠️  Colonna testo assente — feature testuali saranno 0')

    TIMESTAMP_OK = False
    if COL_TIME is not None:
        try:
            df_raw[COL_TIME] = pd.to_datetime(df_raw[COL_TIME], errors='coerce', utc=True)
            df_raw = df_raw.sort_values([COL_USER_ID, COL_TIME])
            TIMESTAMP_OK = True
            print(f'✅ Timestamp: {df_raw[COL_TIME].notna().sum():,} validi')
        except Exception as e:
            print(f'⚠️  Errore timestamp: {e}')

    # ── Fit TF-IDF per coerenza coseno ────────────────────────────────────────
    tfidf_aggr = TfidfVectorizer(max_features=3000, sublinear_tf=True,
                                 min_df=2, token_pattern=r'(?u)\b\w+\b')
    tfidf_aggr.fit(df_raw[COL_TEXT].fillna('').tolist())

    # ── Aggregazione ──────────────────────────────────────────────────────────
    records = []
    gruppi  = list(df_raw.groupby(COL_USER_ID))
    N       = len(gruppi)

    for idx, (uid, grp) in enumerate(gruppi):
        if idx % 500 == 0:
            print(f'   {idx:,}/{N:,}...', end='\r')

        testi   = grp[COL_TEXT].tolist()
        corpus  = ' '.join(testi)
        n_tw    = len(testi)
        lungh   = [len(t) for t in testi]
        l_mu    = float(np.mean(lungh))
        l_sg    = float(np.std(lungh)) if n_tw > 1 else 0.0

        # ── Timestamp ────────────────────────────────────────────────────────
        if TIMESTAMP_OK and COL_TIME in grp.columns:
            ts = sorted(grp[COL_TIME].dropna().tolist())
            freq_g = n_tw / max((ts[-1]-ts[0]).total_seconds()/86400, 1e-6) if len(ts)>=2 else 1.0
            cv_i   = _cv_iat(ts)
            ent_h  = _entropia_oraria([t.hour for t in ts])
        else:
            freq_g = cv_i = ent_h = 0.0

        # ── Social ────────────────────────────────────────────────────────────
        if COL_FOLLOWER and COL_FOLLOWING:
            try:
                foll_r = float(grp[COL_FOLLOWER].iloc[0]) / (float(grp[COL_FOLLOWING].iloc[0]) + 1.0)
            except Exception:
                foll_r = 0.0
        else:
            foll_r = 0.0

        if COL_RETWEET:
            rt_med = pd.to_numeric(grp[COL_RETWEET], errors='coerce').mean()
            rt_med = float(rt_med) if not pd.isna(rt_med) else 0.0
        else:
            rt_med = float(np.mean([1 if str(t).startswith('RT @') else 0 for t in testi]))

        cos_mu, cos_sg = _coerenza_coseno(testi, tfidf_aggr)

        records.append({
            'user_id':            uid,
            'is_bot':             int(grp['is_bot'].mode()[0]),
            'n_tweet':            n_tw,
            'lung_media':         l_mu,
            'lung_std':           l_sg,
            'url_ratio':          _ratio_prefisso(testi, 'http'),
            'hash_ratio':         _ratio_prefisso(testi, '#'),
            'ment_ratio':         _ratio_prefisso(testi, '@'),
            'non_ascii_ratio':    float(np.mean([_non_ascii(t) for t in testi])),
            'entropia_lessicale': float(np.mean([_entropia_shannon(t) for t in testi])),
            'ttr':                _ttr(corpus),
            'freq_giornaliera':   freq_g,
            'cv_iat':             cv_i,
            'entropia_oraria':    ent_h,
            'cv_lunghezza':       l_sg / l_mu if l_mu > 0 else 0.0,
            'foll_ratio':         foll_r,
            'rt_medio':           rt_med,
            'rapporto_unici':     len(set(testi)) / n_tw if n_tw > 0 else 1.0,
            'cos_media':          cos_mu,
            'cos_std':            cos_sg,
        })

    df_utenti = pd.DataFrame(records)

else:
    # ── Dati già user-level ───────────────────────────────────────────────────
    print('ℹ️  Dati già user-level — nessuna aggregazione necessaria')
    if COL_USER_ID and COL_USER_ID != 'user_id':
        df_raw = df_raw.rename(columns={COL_USER_ID: 'user_id'})
    elif COL_USER_ID is None:
        df_raw = df_raw.reset_index()
        df_raw['user_id'] = df_raw.index
    TIMESTAMP_OK = False
    col_num = [c for c in df_raw.select_dtypes(include=[np.number]).columns
               if c not in ['user_id', 'is_bot']]
    df_utenti = df_raw[['user_id', 'is_bot'] + col_num].copy()

print(f'\n\n✅ Completato: {len(df_utenti):,} utenti, {df_utenti.shape[1]} colonne')
display(df_utenti.head(3))

# ── Verifica unicità etichetta ────────────────────────────────────────────────
dup = df_utenti.groupby('user_id')['is_bot'].nunique()
if (dup > 1).any():
    raise ValueError(
        f'ERRORE: {(dup > 1).sum()} utenti con etichette is_bot inconsistenti.'
    )
print('✅ Unicità etichetta is_bot per utente verificata')

## Sezione 3 – Split Stratificato

Divisione 70% train / 30% test con stratificazione su `is_bot` a livello di utente.
Nessun trasformatore viene applicato in questa fase. Lo split avviene prima
di qualsiasi normalizzazione per evitare data leakage.

In [ ]:
COLONNE_FEATURE = [c for c in df_utenti.columns if c not in ['user_id', 'is_bot']]
X_all      = df_utenti[COLONNE_FEATURE].copy()
y_all      = df_utenti['is_bot'].astype(int).values
uid_all    = df_utenti['user_id'].values

print(f'Feature disponibili ({len(COLONNE_FEATURE)}): {COLONNE_FEATURE}')

X_train, X_test, y_train, y_test, uid_train, uid_test = train_test_split(
    X_all, y_all, uid_all,
    test_size=0.30,
    stratify=y_all,
    random_state=RANDOM_STATE
)

print(f'\n✅ Split:  Train {X_train.shape[0]:,} utenti | Test {X_test.shape[0]:,} utenti')

for nome, y in [('TRAIN', y_train), ('TEST', y_test)]:
    uniq, cnt = np.unique(y, return_counts=True)
    print(f'\n📌 DISTRIBUZIONE is_bot nel {nome}:')
    for u, c in zip(uniq, cnt):
        print(f"   {'Bot' if u==1 else 'Umano':6s} ({u}): {c:,} ({c/len(y):.1%})")

rapporto_bot_train = float(np.mean(y_train))
DATASET_SBILANCIATO = rapporto_bot_train < 0.30 or rapporto_bot_train > 0.70

if DATASET_SBILANCIATO:
    print(f'\n⚠️  Dataset sbilanciato (bot={rapporto_bot_train:.2%})')
    print('   → class_weight="balanced" + SMOTE (se disponibile)')
else:
    print(f'\nℹ️  Dataset bilanciato (bot={rapporto_bot_train:.2%})')

## Sezione 4 – Feature Engineering e Preprocessing

Imputazione (mediana) e scalatura robusta (`RobustScaler`) fittate **solo sul train set**
e applicate anche al test. Visualizzazione della matrice di correlazione.
Selezione automatica delle top-20 feature con `SelectKBest(f_classif)`.

In [ ]:
print('⏳ Inizio: preprocessing (imputer + scaler + selector)...')

k_feat = min(20, X_train.shape[1])

preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
])
selector = SelectKBest(score_func=f_classif, k=k_feat)

X_train_prep = preprocessor.fit_transform(X_train, y_train)
X_test_prep  = preprocessor.transform(X_test)

X_train_sel  = selector.fit_transform(X_train_prep, y_train)
X_test_sel   = selector.transform(X_test_prep)

mask_sel         = selector.get_support()
nomi_feature_sel = np.array(COLONNE_FEATURE)[mask_sel].tolist()

print(f'✅ Completato: train {X_train_sel.shape} | test {X_test_sel.shape}')

# ── Top feature per F-score ───────────────────────────────────────────────────
punteggi = selector.scores_[mask_sel]
df_fsel = pd.DataFrame({'feature': nomi_feature_sel, 'f_score': punteggi}) \
            .sort_values('f_score', ascending=False)
print(f'\n📌 TOP-{k_feat} FEATURE SELEZIONATE:')
print(df_fsel.to_string(index=False))

# ── SMOTE se sbilanciato ──────────────────────────────────────────────────────
if DATASET_SBILANCIATO and SMOTE_OK:
    print('\n⏳ Applicazione SMOTE...')
    sm = SMOTE(random_state=RANDOM_STATE)
    X_train_sel, y_train = sm.fit_resample(X_train_sel, y_train)
    print(f'✅ SMOTE: nuova shape train {X_train_sel.shape}')
    print(f'   Distribuzione: {dict(zip(*np.unique(y_train, return_counts=True)))}')

print('\n✅ Preprocessing completato')

In [ ]:
# ── Matrice di correlazione ───────────────────────────────────────────────────
df_corr = pd.DataFrame(X_train_sel, columns=nomi_feature_sel)
df_corr['is_bot'] = y_train[:len(df_corr)]

fig, ax = plt.subplots(figsize=(max(10, len(nomi_feature_sel)),
                                max(8,  len(nomi_feature_sel))))
cm = df_corr.corr()
mask = np.triu(np.ones_like(cm, dtype=bool))
sns.heatmap(cm, mask=mask, annot=len(nomi_feature_sel) <= 14,
            fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, linewidths=0.5)
ax.set_title('Matrice di correlazione delle feature selezionate', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Matrice di correlazione salvata')

# ── Bar chart F-score ─────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, max(5, len(nomi_feature_sel)*0.4)))
df_fsel.sort_values('f_score').plot.barh(x='feature', y='f_score',
                                          ax=ax2, color='steelblue', legend=False)
ax2.set_title(f'Top-{k_feat} Feature — F-score ANOVA (SelectKBest)', fontsize=13)
ax2.set_xlabel('F-score')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_selection.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Grafico feature selection salvato')

## Sezione 5 – Training di Sette Modelli

Addestramento sul train set preprocessato di:
1. Logistic Regression · 2. Random Forest · 3. XGBoost · 4. LightGBM ·
5. SVM RBF · 6. Isolation Forest (non supervisionato) · 7. Ensemble Voting.

Per i modelli supervisionati viene eseguita cross-validazione 5-fold (F1-macro).

In [ ]:
# ── Configurazione class_weight ───────────────────────────────────────────────
CW = 'balanced' if DATASET_SBILANCIATO else None
spw = (float(np.sum(y_train==0)) / max(float(np.sum(y_train==1)), 1.0)
       if DATASET_SBILANCIATO else 1.0)

print(f'class_weight: {CW}  |  scale_pos_weight (XGB): {spw:.2f}')

# ── Definizione modelli ───────────────────────────────────────────────────────
lr_m = LogisticRegression(
    max_iter=1000, class_weight=CW,
    random_state=RANDOM_STATE, solver='lbfgs', C=1.0)

rf_m = RandomForestClassifier(
    n_estimators=300,
    class_weight=CW if CW else 'balanced_subsample',
    random_state=RANDOM_STATE, n_jobs=-1)

xgb_m = XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=6,
    subsample=0.8, random_state=RANDOM_STATE,
    use_label_encoder=False, eval_metric='logloss',
    scale_pos_weight=spw)

lgbm_m = LGBMClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=6,
    class_weight=CW, random_state=RANDOM_STATE, verbose=-1)

svm_m = SVC(
    kernel='rbf', probability=True,
    class_weight=CW, random_state=RANDOM_STATE)

iso_m = IsolationForest(
    n_estimators=200, contamination='auto', random_state=RANDOM_STATE)

ens_m = VotingClassifier(
    estimators=[('lr', lr_m), ('rf', rf_m), ('xgb', xgb_m), ('lgbm', lgbm_m)],
    voting='soft', n_jobs=-1)

MODELLI_SUP = {
    'Logistic Regression': lr_m,
    'Random Forest':       rf_m,
    'XGBoost':             xgb_m,
    'LightGBM':            lgbm_m,
    'SVM RBF':             svm_m,
    'Ensemble Voting':     ens_m,
}
print('✅ Modelli configurati')

In [ ]:
print('⏳ Inizio: cross-validazione 5-fold...')

cv_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_risultati = {}

for nome, mod in MODELLI_SUP.items():
    print(f'   {nome}...', end=' ', flush=True)
    scores = cross_val_score(mod, X_train_sel, y_train,
                             cv=cv_skf, scoring='f1_macro', n_jobs=-1)
    cv_risultati[nome] = scores
    print(f'F1-macro = {scores.mean():.4f} ± {scores.std():.4f}')

print('\n✅ Completato: cross-validazione')

In [ ]:
print('⏳ Inizio: addestramento finale su tutto il train set...')

modelli_addestrati = {}

for nome, mod in MODELLI_SUP.items():
    print(f'   Training {nome}...', end=' ', flush=True)
    mod.fit(X_train_sel, y_train)
    modelli_addestrati[nome] = mod
    print('✅')

# ── Isolation Forest (non supervisionato — non vede y_train) ──────────────────
print('   Training Isolation Forest (non supervisionato)...', end=' ', flush=True)
iso_m.fit(X_train_sel)   # SOLO X, no etichette
modelli_addestrati['Isolation Forest'] = iso_m

# Soglia ottimale su train via massimizzazione F1-macro
sc_tr = -(iso_m.score_samples(X_train_sel))
sc_tr_n = (sc_tr - sc_tr.min()) / (sc_tr.max() - sc_tr.min() + 1e-12)
SOGLIA_ISO = 0.5
best_f1_iso = 0.0
for th in np.linspace(0.1, 0.9, 81):
    f1_tmp = f1_score(y_train, (sc_tr_n >= th).astype(int),
                      average='macro', zero_division=0)
    if f1_tmp > best_f1_iso:
        best_f1_iso, SOGLIA_ISO = f1_tmp, th

print(f'✅ (soglia={SOGLIA_ISO:.2f}, F1-macro train={best_f1_iso:.4f})')
print('\n✅ Completato: addestramento tutti i modelli')

## Sezione 6 – Valutazione sul Test Set

Per ogni modello: classification report, matrice di confusione, curve ROC e
Precision-Recall. Tutti i risultati confluiscono in `results_df` ordinato per F1-macro.

In [ ]:
print('⏳ Inizio: valutazione sul test set...')

risultati_lista = []
score_per_roc   = {}
PALETTE         = plt.cm.tab10.colors

n_mod    = len(modelli_addestrati)
n_cols_c = 3
n_rows_c = math.ceil(n_mod / n_cols_c)
fig_cf, assi_cf = plt.subplots(n_rows_c, n_cols_c,
                                figsize=(6*n_cols_c, 5*n_rows_c))
assi_cf = np.array(assi_cf).flatten()

for idx, (nome, mod) in enumerate(modelli_addestrati.items()):
    print(f'\n{"="*50}\n📊 {nome}\n{"="*50}')

    if nome == 'Isolation Forest':
        sc_raw  = -(mod.score_samples(X_test_sel))
        y_prob  = (sc_raw - sc_raw.min()) / (sc_raw.max() - sc_raw.min() + 1e-12)
        y_pred  = (y_prob >= SOGLIA_ISO).astype(int)
    elif hasattr(mod, 'predict_proba'):
        y_prob  = mod.predict_proba(X_test_sel)[:, 1]
        y_pred  = mod.predict(X_test_sel)
    else:
        yd      = mod.decision_function(X_test_sel)
        y_prob  = (yd - yd.min()) / (yd.max() - yd.min() + 1e-12)
        y_pred  = mod.predict(X_test_sel)

    score_per_roc[nome] = y_prob

    acc   = accuracy_score(y_test, y_pred)
    f1_m  = f1_score(y_test, y_pred, average='macro', zero_division=0)
    try:    roc_a = roc_auc_score(y_test, y_prob)
    except: roc_a = float('nan')
    ap    = average_precision_score(y_test, y_prob)

    print(f'Accuracy={acc:.4f} | F1-macro={f1_m:.4f} | ROC-AUC={roc_a:.4f} | AP={ap:.4f}')
    print(classification_report(y_test, y_pred,
                                 target_names=['Umano(0)', 'Bot(1)'],
                                 zero_division=0))
    risultati_lista.append({
        'model': nome, 'accuracy': acc, 'f1_macro': f1_m,
        'roc_auc': roc_a, 'avg_precision': ap
    })

    cm_val = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Umano','Bot'], yticklabels=['Umano','Bot'],
                ax=assi_cf[idx])
    assi_cf[idx].set_title(nome, fontsize=11)
    assi_cf[idx].set_xlabel('Predetto'); assi_cf[idx].set_ylabel('Reale')

for j in range(idx+1, len(assi_cf)):
    assi_cf[j].set_visible(False)

fig_cf.suptitle('Matrici di Confusione — Tutti i Modelli', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n✅ Completato: valutazione modelli')

In [ ]:
# ── Curve ROC ─────────────────────────────────────────────────────────────────
fig_roc, ax_roc = plt.subplots(figsize=(10, 8))
for i, (nome, yp) in enumerate(score_per_roc.items()):
    try:
        fpr, tpr, _ = roc_curve(y_test, yp)
        rauc = auc(fpr, tpr)
        ax_roc.plot(fpr, tpr, color=PALETTE[i%10], lw=2,
                    label=f'{nome} (AUC={rauc:.3f})')
    except Exception as e:
        print(f'⚠️  ROC non calcolabile per {nome}: {e}')
ax_roc.plot([0,1],[0,1],'k--',lw=1.5,label='Casuale')
ax_roc.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
           title='Curve ROC — Confronto Modelli')
ax_roc.legend(loc='lower right', fontsize=10); ax_roc.grid(True,alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'roc_curves.png', dpi=120, bbox_inches='tight')
plt.show(); print('✅ Curve ROC salvate')

# ── Curve Precision-Recall ──────────────────────────────────────────────────
fig_pr, ax_pr = plt.subplots(figsize=(10, 8))
for i, (nome, yp) in enumerate(score_per_roc.items()):
    try:
        prec, rec, _ = precision_recall_curve(y_test, yp)
        ap = average_precision_score(y_test, yp)
        ax_pr.plot(rec, prec, color=PALETTE[i%10], lw=2,
                   label=f'{nome} (AP={ap:.3f})')
    except Exception as e:
        print(f'⚠️  PR non calcolabile per {nome}: {e}')
br = np.mean(y_test)
ax_pr.axhline(y=br, color='k', linestyle='--', lw=1.5,
              label=f'Base rate ({br:.2f})')
ax_pr.set(xlabel='Recall', ylabel='Precision',
          title='Curve Precision-Recall — Confronto Modelli')
ax_pr.legend(loc='upper right', fontsize=10); ax_pr.grid(True,alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'pr_curves.png', dpi=120, bbox_inches='tight')
plt.show(); print('✅ Curve PR salvate')

# ── DataFrame riassuntivo ────────────────────────────────────────────────────
results_df = (pd.DataFrame(risultati_lista)
              .sort_values('f1_macro', ascending=False)
              .reset_index(drop=True))
print('\n📌 RIEPILOGO RISULTATI (ordinato per F1-macro):')
display(results_df)

## Sezione 7 – Interpretabilità

Feature importance (o coefficienti) del miglior modello per F1-macro.
Calcolo dei valori SHAP (se disponibili) con beeswarm plot delle top-20 feature.
Analisi qualitativa di 5 falsi positivi e 5 falsi negativi.

In [ ]:
MIGLIOR_NOME    = results_df.iloc[0]['model']
MIGLIOR_MODELLO = modelli_addestrati[MIGLIOR_NOME]
print(f'🏆 Miglior modello: {MIGLIOR_NOME}')
print(f'   F1-macro: {results_df.iloc[0]["f1_macro"]:.4f}')
print(f'   ROC-AUC:  {results_df.iloc[0]["roc_auc"]:.4f}')

In [ ]:
print('⏳ Inizio: feature importance...')

fig_fi, ax_fi = plt.subplots(figsize=(10, max(5, k_feat*0.4)))

if hasattr(MIGLIOR_MODELLO, 'feature_importances_'):
    imp = MIGLIOR_MODELLO.feature_importances_
    titolo_fi = 'Feature Importance (Gini)'
elif hasattr(MIGLIOR_MODELLO, 'coef_'):
    imp = np.abs(MIGLIOR_MODELLO.coef_[0])
    titolo_fi = '|Coefficiente| — Logistic Regression'
elif hasattr(MIGLIOR_MODELLO, 'estimators_'):
    imp = None
    for nme, est in MIGLIOR_MODELLO.estimators:
        if hasattr(est, 'feature_importances_'):
            imp = est.feature_importances_
            titolo_fi = f'Feature Importance RF (interno a {MIGLIOR_NOME})'
            break
    if imp is None:
        imp = np.ones(len(nomi_feature_sel)); titolo_fi='Non disponibile'
else:
    imp = np.ones(len(nomi_feature_sel)); titolo_fi='Non disponibile'

n_fi = min(len(imp), len(nomi_feature_sel))
df_imp = pd.DataFrame({'feature': nomi_feature_sel[:n_fi], 'importanza': imp[:n_fi]}) \
           .sort_values('importanza', ascending=True)

df_imp.plot.barh(x='feature', y='importanza', ax=ax_fi,
                 color='teal', legend=False)
ax_fi.set_title(f'{MIGLIOR_NOME} — {titolo_fi}', fontsize=13)
ax_fi.set_xlabel('Importanza')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'feature_importances.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Completato: feature importance')

In [ ]:
print('⏳ Inizio: calcolo SHAP values...')

X_test_df  = pd.DataFrame(X_test_sel,  columns=nomi_feature_sel)
X_train_df = pd.DataFrame(X_train_sel[:min(500, len(X_train_sel))],
                           columns=nomi_feature_sel)

if SHAP_OK:
    try:
        if isinstance(MIGLIOR_MODELLO, (RandomForestClassifier,)):
            explainer = shap.TreeExplainer(MIGLIOR_MODELLO)
            sv = explainer.shap_values(X_test_df)
            shap_vals = sv[1] if isinstance(sv, list) else sv
        elif isinstance(MIGLIOR_MODELLO, LogisticRegression):
            explainer = shap.LinearExplainer(MIGLIOR_MODELLO, X_train_df)
            shap_vals = explainer.shap_values(X_test_df)
        elif hasattr(MIGLIOR_MODELLO, '_m'):          # shim XGB/LGBM
            explainer = shap.TreeExplainer(MIGLIOR_MODELLO._m)
            sv = explainer.shap_values(X_test_df)
            shap_vals = sv[1] if isinstance(sv, list) else sv
        else:
            bg = shap.kmeans(X_train_df, 20)
            def _fn(x): return MIGLIOR_MODELLO.predict_proba(x)[:,1]
            explainer = shap.KernelExplainer(_fn, bg)
            shap_vals = explainer.shap_values(X_test_df.iloc[:100], nsamples=100)
            X_test_df = X_test_df.iloc[:100]

        shap_exp = shap.Explanation(
            values=shap_vals,
            data=X_test_df.values,
            feature_names=nomi_feature_sel
        )
        plt.figure(figsize=(12, 8))
        shap.plots.beeswarm(shap_exp, max_display=20, show=False)
        plt.title(f'SHAP Beeswarm — {MIGLIOR_NOME}', fontsize=13)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR/'shap_beeswarm.png', dpi=120, bbox_inches='tight')
        plt.show()
        print('✅ Completato: SHAP beeswarm salvato')
        shap_vals_saved = shap_vals
    except Exception as e:
        print(f'⚠️  SHAP non calcolabile per {MIGLIOR_NOME}: {e}')
        shap_vals_saved = None
else:
    print('⚠️  SHAP non installato — sezione saltata')
    shap_vals_saved = None

In [ ]:
print('⏳ Inizio: analisi falsi positivi / negativi...')

# Predizioni miglior modello
if MIGLIOR_NOME == 'Isolation Forest':
    sc_b  = -(MIGLIOR_MODELLO.score_samples(X_test_sel))
    sc_bn = (sc_b - sc_b.min()) / (sc_b.max() - sc_b.min() + 1e-12)
    y_pred_best = (sc_bn >= SOGLIA_ISO).astype(int)
else:
    y_pred_best = MIGLIOR_MODELLO.predict(X_test_sel)

df_err = pd.DataFrame(X_test_sel, columns=nomi_feature_sel)
df_err['y_vero']    = y_test
df_err['y_predetto']= y_pred_best
df_err['user_id']   = uid_test

FP = df_err[(df_err.y_vero==0) & (df_err.y_predetto==1)]
FN = df_err[(df_err.y_vero==1) & (df_err.y_predetto==0)]

print(f'\n   Falsi Positivi (umani → bot):  {len(FP):,}')
print(f'   Falsi Negativi (bot → umani): {len(FN):,}')

if len(FP) > 0:
    print('\n📌 PROFILO DEI 5 FALSI POSITIVI (umani classificati come bot):')
    display(FP.drop(columns=['y_vero','y_predetto']).head(5))
    print("""
💬 COMMENTO — Falsi Positivi:
Gli utenti umani classificati erroneamente come bot mostrano pattern atipici:
alta frequenza di posting (giornalisti, account istituzionali, social media manager),
uso sistematico di URL e hashtag, o testi brevi e ripetitivi (customer care, risposte
automatizzate legittime). Il modello interpreta questi pattern come segnali di
automazione, confondendo professionalità operativa con comportamento bot.
    """)

if len(FN) > 0:
    print('\n📌 PROFILO DEI 5 FALSI NEGATIVI (bot classificati come umani):')
    display(FN.drop(columns=['y_vero','y_predetto']).head(5))
    print("""
💬 COMMENTO — Falsi Negativi:
I bot classificati come umani sono probabilmente bot di nuova generazione
("social bots", Cresci et al. 2020): imitano la variabilità temporale
(alto CV-IAT), la ricchezza lessicale (TTR elevato) e mantengono bassa
autosimilarità coseno tra i messaggi. Questi bot superano i filtri basati
su singole feature perché riproducono la complessità stocastica degli
utenti reali su brevi orizzonti temporali.
    """)

print('✅ Completato: analisi errori')

## Sezione 8 – Analisi degli Errori e Pattern Nascosti

Clustering K-Means dei campioni mal classificati, riduzione UMAP/PCA a 2D.
Analisi temporale (se `created_at` è disponibile).
Box plot delle feature più discriminanti separati per bot/umani.

In [ ]:
print('⏳ Inizio: clustering degli errori...')

mask_err   = y_pred_best != y_test
X_err      = X_test_sel[mask_err]
y_err      = y_test[mask_err]
yp_err     = y_pred_best[mask_err]
n_err      = int(np.sum(mask_err))
print(f'   Campioni mal classificati: {n_err}')

if n_err >= 6:
    tipo_err = np.where((y_err==0) & (yp_err==1), 'Falso Positivo', 'Falso Negativo')
    k_cl     = min(3, n_err - 1)
    km_err   = KMeans(n_clusters=k_cl, random_state=RANDOM_STATE, n_init=10)
    cl_lab   = km_err.fit_predict(X_err)

    if UMAP_OK and n_err >= 10:
        print('   Riduzione con UMAP...', end=' ', flush=True)
        reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE,
                             n_neighbors=min(15, n_err-1))
        X2d  = reducer.fit_transform(X_err)
        met  = 'UMAP'
    else:
        print('   Riduzione con PCA...', end=' ', flush=True)
        reducer = PCA(n_components=2, random_state=RANDOM_STATE)
        X2d  = reducer.fit_transform(X_err)
        met  = 'PCA'
    print('✅')

    fig_cl, ax_cl = plt.subplots(figsize=(10, 7))
    COLORI_TIPO = {'Falso Positivo': 'darkorange', 'Falso Negativo': 'crimson'}
    MARCATORI   = ['o', 's', '^', 'D']

    for c in range(k_cl):
        m_c = cl_lab == c
        for tipo in np.unique(tipo_err):
            m_t = tipo_err == tipo
            m_comb = m_c & m_t
            if m_comb.sum() > 0:
                ax_cl.scatter(
                    X2d[m_comb, 0], X2d[m_comb, 1],
                    c=COLORI_TIPO[tipo],
                    marker=MARCATORI[c % len(MARCATORI)],
                    label=f'Cluster {c} — {tipo}',
                    s=80, alpha=0.8, edgecolors='white', linewidths=0.5
                )

    ax_cl.set(title=f'Clustering Errori ({met} 2D + K-Means k={k_cl})',
              xlabel=f'{met} 1', ylabel=f'{met} 2')
    ax_cl.legend(fontsize=9); ax_cl.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'error_clustering.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('✅ Completato: clustering errori salvato')
else:
    print(f'⚠️  Troppo pochi errori ({n_err}) per clustering — saltato')

In [ ]:
# ── Analisi temporale ────────────────────────────────────────────────────────
if (IS_ROW_LEVEL and
    'TIMESTAMP_OK' in dir() and TIMESTAMP_OK and
    COL_TIME is not None):

    print('⏳ Inizio: analisi temporale mensile...')
    df_date_u = df_raw.groupby(COL_USER_ID)[COL_TIME].min().reset_index()
    df_date_u.columns = ['user_id', 'data_primo']
    df_date_u['data_primo'] = pd.to_datetime(df_date_u['data_primo'],
                                              errors='coerce', utc=True)

    df_tmp = pd.DataFrame({
        'user_id': uid_test, 'y_vero': y_test, 'y_pred': y_pred_best
    }).merge(df_date_u, on='user_id', how='left').dropna(subset=['data_primo'])

    if len(df_tmp) > 20:
        df_tmp['mese'] = df_tmp['data_primo'].dt.to_period('M')
        mesi = sorted(df_tmp['mese'].unique())
        f1_mesi = []
        for m in mesi:
            sub = df_tmp[df_tmp['mese'] == m]
            if len(sub) >= 5:
                f1_mesi.append({
                    'mese': str(m),
                    'f1_macro': f1_score(sub['y_vero'], sub['y_pred'],
                                         average='macro', zero_division=0),
                    'n': len(sub)
                })
        if f1_mesi:
            df_t = pd.DataFrame(f1_mesi)
            fig_t, ax_t = plt.subplots(figsize=(12, 5))
            ax_t.plot(df_t['mese'], df_t['f1_macro'], marker='o',
                      color='steelblue', lw=2)
            ax_t.set(title='F1-macro mensile sul Test Set',
                     xlabel='Mese', ylabel='F1-macro')
            ax_t.tick_params(axis='x', rotation=45)
            ax_t.grid(True, alpha=0.4)
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR/'temporal_f1.png', dpi=120, bbox_inches='tight')
            plt.show()
            print('✅ Completato: analisi temporale salvata')
        else:
            print('⚠️  Nessun mese con ≥5 campioni — analisi temporale saltata')
    else:
        print('⚠️  Dati temporali insufficienti nel test set')
else:
    print('ℹ️  Colonna created_at non disponibile — analisi temporale saltata')

In [ ]:
# ── Box plot feature discriminanti ──────────────────────────────────────────
print('⏳ Inizio: box plot feature discriminanti...')

df_bp = pd.DataFrame(X_test_sel, columns=nomi_feature_sel)
df_bp['is_bot']  = y_test
df_bp['Classe']  = df_bp['is_bot'].map({0:'Umano', 1:'Bot'})

top6 = df_fsel.sort_values('f_score', ascending=False)['feature'].head(6).tolist()
top6 = [f for f in top6 if f in df_bp.columns]

if top6:
    n_b = len(top6)
    fig_bp, assi_bp = plt.subplots(2, math.ceil(n_b/2), figsize=(14, 8))
    assi_bp = np.array(assi_bp).flatten()
    for i, feat in enumerate(top6):
        sns.boxplot(data=df_bp, x='Classe', y=feat,
                    palette={'Umano':'steelblue','Bot':'tomato'},
                    ax=assi_bp[i])
        assi_bp[i].set_title(feat, fontsize=11)
        assi_bp[i].set_xlabel('')
    for j in range(i+1, len(assi_bp)):
        assi_bp[j].set_visible(False)
    fig_bp.suptitle('Feature Discriminanti — Bot vs Umani', fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'feature_boxplots.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('✅ Completato: box plot salvati')
else:
    print('⚠️  Nessuna feature per i box plot')

## Sezione 9 – Salvataggio Artefatti

Salvataggio in `/home/cesare/Scrivania/Vannacci/mib_output/` di: miglior modello,
pipeline di preprocessing, DataFrame risultati e tutti i grafici prodotti.
Riepilogo finale in italiano con i risultati chiave.

In [ ]:
print('⏳ Inizio: salvataggio artefatti...')

# Miglior modello
joblib.dump(MIGLIOR_MODELLO, OUTPUT_DIR/'best_model.joblib')
print(f'✅ Miglior modello salvato: {OUTPUT_DIR/"best_model.joblib"}')

# Preprocessor bundle
bundle = {
    'preprocessor':      preprocessor,
    'selector':          selector,
    'feature_names_in':  COLONNE_FEATURE,
    'feature_names_out': nomi_feature_sel,
}
joblib.dump(bundle, OUTPUT_DIR/'preprocessor.joblib')
print(f'✅ Preprocessor salvato: {OUTPUT_DIR/"preprocessor.joblib"}')

# Risultati CSV
results_df.to_csv(OUTPUT_DIR/'results.csv', index=False)
print(f'✅ Risultati CSV salvati: {OUTPUT_DIR/"results.csv"}')

print('\n📁 ARTEFATTI IN:', OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'   {f.name:40s} ({f.stat().st_size/1024:.1f} KB)')

print('\n✅ Completato: salvataggio artefatti')

In [ ]:
# ── Riepilogo finale ─────────────────────────────────────────────────────────
print('\n' + '═'*65)
print(' RIEPILOGO FINALE — MIB Social Bot Detection')
print('═'*65)
print(f'\n📂 Dataset:           {PERCORSO_PARQUET.name}')
print(f'👥 Utenti totali:     {len(df_utenti):,}')
print(f'   Train (70%):       {X_train.shape[0]:,}')
print(f'   Test  (30%):       {X_test.shape[0]:,}')
print(f'\n⚖️  Sbilanciato:      {"Sì" if DATASET_SBILANCIATO else "No"} '
      f'(bot train={rapporto_bot_train:.2%})')
print(f'\n🔑 Feature ingegnerizzate:  {len(COLONNE_FEATURE)}')
print(f'🎯 Feature selezionate:     {k_feat}')
print(f'   → {nomi_feature_sel}')

print('\n📊 RISULTATI (ordinati per F1-macro):')
print(f'{"Modello":<26} {"Accuracy":>10} {"F1-macro":>10} {"ROC-AUC":>10} {"Avg Prec":>10}')
print('-'*66)
for _, r in results_df.iterrows():
    roc_str = f'{r["roc_auc"]:.4f}' if not (isinstance(r["roc_auc"], float)
                                              and math.isnan(r["roc_auc"])) else '  N/A  '
    print(f'{r["model"]:<26} {r["accuracy"]:>10.4f} {r["f1_macro"]:>10.4f} '
          f'{roc_str:>10} {r["avg_precision"]:>10.4f}')

br = results_df.iloc[0]
print(f'\n{"─"*65}')
print(f'🏆 MIGLIOR MODELLO: {MIGLIOR_NOME}')
print(f'   F1-macro:         {br["f1_macro"]:.4f}')
rstr = f'{br["roc_auc"]:.4f}' if not math.isnan(br["roc_auc"]) else 'N/A'
print(f'   ROC-AUC:          {rstr}')
print(f'   Avg Precision:    {br["avg_precision"]:.4f}')
print(f'\n📁 Output:  {OUTPUT_DIR}')
print("""
ℹ️  NOTA METODOLOGICA:
   Tutti i transformer (TF-IDF, RobustScaler, SelectKBest) sono stati
   fittati esclusivamente sul train set per prevenire data leakage.
   Isolation Forest non ha mai visto le etichette is_bot durante
   l'addestramento, simulando un rilevamento anomalie puro.
   Approccio basato su Cresci et al. — ICWSM 2017/2020.
""")
print('═'*65)